In [25]:
import numpy as np, json
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras import optimizers
from tensorflow.keras.layers import LeakyReLU, BatchNormalization
from sklearn.metrics import classification_report, f1_score

In [26]:
data = np.load("preprocessed_arrays.npz")
X_train_pad, y_train_cat = data["X_train_pad"], data["y_train_cat"]
X_val_pad,   y_val_cat   = data["X_val_pad"],   data["y_val_cat"]

with open("label_encoder_classes.json","r",encoding="utf-8") as f:
    classes = json.load(f)

with open("tokenizer.json","r",encoding="utf-8") as f:
    tok = tokenizer_from_json(f.read())

MAX_LEN   = X_train_pad.shape[1]
MAX_VOCAB = tok.num_words or (len(tok.word_index) + 1)

In [27]:
model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN, mask_zero=False),
    SpatialDropout1D(0.15),

    Conv1D(filters=64, kernel_size=2, activation="relu", padding="same"),
    Conv1D(filters=64, kernel_size=2, activation="relu", padding="same"),

    LSTM(64, dropout=0.3, recurrent_dropout=0.2),

    Dropout(0.3),
    Dense(64),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.25),

    Dense(len(classes), activation="softmax")
])
opt = optimizers.Adam(learning_rate=2e-4, clipnorm=1.0)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
model.summary()

c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_4             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
cb = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)
]

history = model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=30,
    batch_size=128,
    callbacks=cb,
    verbose=1
)

Epoch 1/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 56s 205ms/step - accuracy: 0.3368 - loss: 1.1039 - val_accuracy: 0.3342 - val_loss: 1.0986 - learning_rate: 2.0000e-04
Epoch 2/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 35s 186ms/step - accuracy: 0.3421 - loss: 1.0979 - val_accuracy: 0.3333 - val_loss: 1.0979 - learning_rate: 2.0000e-04
Epoch 3/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 35s 188ms/step - accuracy: 0.3344 - loss: 1.0973 - val_accuracy: 0.3405 - val_loss: 1.0966 - learning_rate: 2.0000e-04
Epoch 4/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 35s 187ms/step - accuracy: 0.3345 - loss: 1.0955 - val_accuracy: 0.3388 - val_loss: 1.0956 - learning_rate: 2.0000e-04
Epoch 5/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 37s 198ms/step - accuracy: 0.3354 - loss: 1.0946 - val_accuracy: 0.3448 - val_loss: 1.0945 - learning_rate: 2.0000e-04
Epoch 6/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 36s 193ms/step - accuracy: 0.4000 - loss: 1.0717 - val_accuracy: 0.5732 - val_loss: 0.9990 - learning_rate: 2.0000e-04
Epoch 7/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 34s 18

In [29]:
y_true = y_val_cat.argmax(axis=1)
y_pred = model.predict(X_val_pad, verbose=0).argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=classes))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

              precision    recall  f1-score   support

     negatif       0.55      0.69      0.61      2000
        notr       0.75      0.58      0.65      2000
     pozitif       0.79      0.78      0.79      2000

    accuracy                           0.68      6000
   macro avg       0.70      0.68      0.69      6000
weighted avg       0.70      0.68      0.69      6000

Macro F1: 0.6854661822096703


In [30]:
def predict_text(text, model=model, tok=tok, classes=classes, maxlen=MAX_LEN):
    seq  = tok.texts_to_sequences([text])
    pad  = pad_sequences(seq, maxlen=maxlen, padding="post", truncating="post")
    probs = model.predict(pad, verbose=0)[0]
    cls_idx = int(probs.argmax())
    return classes[cls_idx], probs

examples = [
    "Ürün gerçekten harika, çok memnun kaldım.",
    "Kargo çok geç geldi ve paket yırtıktı.",
    "Ürün fena değil, idare eder.",
    "Telefonun özellikleri iyi ama bataryası çabuk bitiyor.",
    "Ürün güzel ama fiyatına göre performansı düşük.",
    "Kargo biraz geç geldi ama satıcı çok yardımcı oldu.",
    "Sevdim ürünü, tekrar alırım.",
    "Tavsiye ederim.",
    "Berbat"
]

for ex in examples:
    label, probs = predict_text(ex, model=model, tok=tok, classes=classes, maxlen=100)
    print(f"Metin: {ex}")
    print(f"Tahmin: {label}, Olasılıklar: {probs}\n")

Metin: Ürün gerçekten harika, çok memnun kaldım.
Tahmin: pozitif, Olasılıklar: [0.1994368  0.07897773 0.7215854 ]

Metin: Kargo çok geç geldi ve paket yırtıktı.
Tahmin: negatif, Olasılıklar: [0.6349312  0.21819103 0.14687774]

Metin: Ürün fena değil, idare eder.
Tahmin: notr, Olasılıklar: [0.19097863 0.80194783 0.0070735 ]

Metin: Telefonun özellikleri iyi ama bataryası çabuk bitiyor.
Tahmin: notr, Olasılıklar: [0.19098242 0.80194354 0.00707401]

Metin: Ürün güzel ama fiyatına göre performansı düşük.
Tahmin: notr, Olasılıklar: [0.19099794 0.8019254  0.00707662]

Metin: Kargo biraz geç geldi ama satıcı çok yardımcı oldu.
Tahmin: notr, Olasılıklar: [0.19097881 0.80194765 0.00707352]

Metin: Sevdim ürünü, tekrar alırım.
Tahmin: pozitif, Olasılıklar: [0.3492681  0.13241565 0.5183162 ]

Metin: Tavsiye ederim.
Tahmin: negatif, Olasılıklar: [0.4262839  0.15798865 0.4157275 ]

Metin: Berbat
Tahmin: negatif, Olasılıklar: [0.49908346 0.18028669 0.3206298 ]



In [31]:
from pathlib import Path

current_dir = Path.cwd() / "Models"
save_path = current_dir / "cnn_lstm_model.keras"

model.save(save_path)
print(f"CNN-LSTM modeli kaydedildi: {save_path}")

CNN-LSTM modeli kaydedildi: c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\Models\cnn_lstm_model.keras
